# 02_modelado — Modelo baseline de riesgo de stunting (16 features basales)

**Entrada unica:** `data/processed/model_dataset.csv` (generado por el Paso 9.2 del EDA).

**Contrato:** 16 features basales (10 maternas + 6 del nacimiento) -> targets `stunted_12` y `stunted_24`. Split 70/15/15 estratificado por bebe; imputacion/escalado SOLO en train; metricas F1, PR-AUC, ROC-AUC y sensibilidad; `class_weight='balanced'`.

**MLOps:** seed fija, hash MD5 del dataset en metadatos y MLflow, modelos en `models/`, figuras en `figures/`.

## 1. Imports y configuracion

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (accuracy_score, precision_score, f1_score, recall_score,
                             roc_auc_score, average_precision_score, confusion_matrix,
                             ConfusionMatrixDisplay, roc_curve, precision_recall_curve)

import joblib
import hashlib
import json
from pathlib import Path

try:
    import mlflow
    import mlflow.sklearn
    MLFLOW_AVAILABLE = True
except ImportError:
    MLFLOW_AVAILABLE = False
    print('[INFO] MLflow no instalado; se omitira el logging.')

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11
sns.set_style('whitegrid')

SEED = 42
np.random.seed(SEED)

import sklearn
print('pandas', pd.__version__)
print('numpy', np.__version__)
print('sklearn', sklearn.__version__)
if MLFLOW_AVAILABLE:
    print('mlflow', mlflow.__version__)

[INFO] MLflow no instalado; se omitira el logging.
pandas 3.0.5
numpy 2.4.6
sklearn 1.9.0


## 2. Rutas y configuracion

In [25]:
def find_root(start=Path.cwd()):
    for p in [start, *start.parents]:
        if (p / '.git').exists() or (p / 'README.md').exists():
            return p
    return start

ROOT = find_root()
DATA_PATH = ROOT / 'data' / 'processed' / 'model_dataset.csv'
FIG_DIR = ROOT / 'figures' / '02_model'        # <- subcarpeta exclusiva de este notebook
MODELS_DIR = ROOT / 'models'
FIG_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

FEATURES_ENROL = ['enrol_hiv_status_cat', 'momage_cat', 'educ_cat_n', 'marital_cat',
                  'hfia_enr', 'wealth_quintile', 'depression', 'mom_muac_cat',
                  'parity', 'enrol_anemia']
FEATURES_NAC = ['b1_sex', 'gestage_final', 'caesarean', 'preterm', 'sga', 'lbw']
FEATURES = FEATURES_ENROL + FEATURES_NAC
TARGETS = ['stunted_12', 'stunted_24']
ID_COL = 'newid'

CONFIG = {'data_path': str(DATA_PATH), 'figures_dir': str(FIG_DIR),
          'models_dir': str(MODELS_DIR), 'n_features': len(FEATURES),
          'targets': TARGETS, 'seed': SEED, 'class_weight': 'balanced'}

print('ROOT:', ROOT)
print('DATA:', DATA_PATH)
print('FIGS:', FIG_DIR)
print('MODELS:', MODELS_DIR)

ROOT: c:\Users\CORE I5 7TH\Documents\Learning\MAIA\IV. Semester\4_1_Bimester\Desarrolo de Soluciones\Grupo_23_Microproyecto\yei_microproyecto-pds-grupo23
DATA: c:\Users\CORE I5 7TH\Documents\Learning\MAIA\IV. Semester\4_1_Bimester\Desarrolo de Soluciones\Grupo_23_Microproyecto\yei_microproyecto-pds-grupo23\data\processed\model_dataset.csv
FIGS: c:\Users\CORE I5 7TH\Documents\Learning\MAIA\IV. Semester\4_1_Bimester\Desarrolo de Soluciones\Grupo_23_Microproyecto\yei_microproyecto-pds-grupo23\figures\02_model
MODELS: c:\Users\CORE I5 7TH\Documents\Learning\MAIA\IV. Semester\4_1_Bimester\Desarrolo de Soluciones\Grupo_23_Microproyecto\yei_microproyecto-pds-grupo23\models


## 3. Carga y validacion del dataset plano

In [3]:
if not DATA_PATH.exists():
    raise FileNotFoundError('No existe ' + str(DATA_PATH) + '. Corre primero el Paso 9.2 del EDA para generar data/processed/model_dataset.csv.')

dataset = pd.read_csv(DATA_PATH)
print('Shape:', dataset.shape)
print('Bebes unicos:', dataset[ID_COL].nunique())
dataset.head()

Shape: (333, 19)
Bebes unicos: 333


,newid,enrol_hiv_status_cat,momage_cat,educ_cat_n,marital_cat,hfia_enr,wealth_quintile,depression,mom_muac_cat,parity,enrol_anemia,b1_sex,gestage_final,caesarean,preterm,sga,lbw,stunted_12,stunted_24
0,96,Negative,26 to 35,Primary or below,Married,severe,quintile2,mild,normal,multi,no,female,38.00,no,no,no,no,no,no
1,97,Positive,35 and more,Secondary and above,Married,secured,highest quintile,no depression,above normal,multi,no,male,39.62,yes,no,no,no,no,no
2,98,Negative,26 to 35,Secondary and above,Married,severe,lowest quintile,no depression,normal,nulli,no,male,38.90,yes,no,no,no,no,no
3,99,Negative,26 to 35,Secondary and above,Married,mild,quintile2,no depression,normal,multi,no,male,43.48,no,no,no,no,no,no
4,100,Negative,25 and less,Secondary and above,Married,secured,highest quintile,no depression,above normal,nulli,no,male,39.19,yes,no,no,no,no,no


In [4]:
required = [ID_COL] + FEATURES + TARGETS
missing_cols = [c for c in required if c not in dataset.columns]
if missing_cols:
    raise ValueError('Faltan columnas en model_dataset.csv: ' + str(missing_cols))
print('Esquema OK:', len(required), 'columnas requeridas presentes.')

miss = dataset[FEATURES].isna().sum()
print('Missing por feature:')
print(miss[miss > 0])

for t in TARGETS:
    print()
    print(t)
    print(dataset[t].value_counts(dropna=False))

Esquema OK: 19 columnas requeridas presentes.
Missing por feature:
hfia_enr         3
parity           2
enrol_anemia    22
caesarean        6
sga             27
lbw              2
dtype: int64

stunted_12
stunted_12
no     277
yes     49
NaN      7
Name: count, dtype: int64

stunted_24
stunted_24
no     228
yes     95
NaN     10
Name: count, dtype: int64


## 4. Definicion de X/y y split 70/15/15 (sin traslape de bebes)

In [6]:
# ============================================================================
# Definicion de X/y y split 70/15/15 (sin traslape de bebes)
# ============================================================================
X_all = dataset[FEATURES].copy()

def make_splits(target_col, seed=SEED):
    """Construye y = target mapeado, descarta bebés sin target en ese horizonte
    y particiona 70/15/15 estratificado (una fila por bebé => sin fuga por grupo)."""
    y = dataset[target_col].map({'no': 0, 'yes': 1})
    mask = y.notna()                                  # <- bebés SIN visita en ese horizonte fuera
    X = X_all[mask]
    y = y[mask].astype(int)
    g = dataset[ID_COL][mask]
    print(f'{target_col}: {len(X)} bebés válidos ({(~mask).sum()} sin target descartados)')

    X_tr, X_tmp, y_tr, y_tmp, g_tr, g_tmp = train_test_split(
        X, y, g, test_size=0.30, stratify=y, random_state=seed)
    X_va, X_te, y_va, y_te, g_va, g_te = train_test_split(
        X_tmp, y_tmp, g_tmp, test_size=0.50, stratify=y_tmp, random_state=seed)

    assert set(g_tr).isdisjoint(g_va)
    assert set(g_tr).isdisjoint(g_te)
    assert set(g_va).isdisjoint(g_te)
    return X_tr, X_va, X_te, y_tr, y_va, y_te, g_tr, g_va, g_te

splits_12 = make_splits('stunted_12')
splits_24 = make_splits('stunted_24')

for name, s in [('stunted_12', splits_12), ('stunted_24', splits_24)]:
    print(name, '| train', len(s[0]), round(s[3].mean() * 100, 1), '% + | val', len(s[1]),
          round(s[4].mean() * 100, 1), '% + | test', len(s[2]), round(s[5].mean() * 100, 1), '% +')

stunted_12: 326 bebés válidos (7 sin target descartados)
stunted_24: 323 bebés válidos (10 sin target descartados)
stunted_12 | train 228 14.9 % + | val 49 16.3 % + | test 49 14.3 % +
stunted_24 | train 226 29.2 % + | val 48 29.2 % + | test 49 30.6 % +


## 5. Pipeline de preprocesamiento (anti-leakage: se ajusta solo en train)

In [15]:
class Winsorizer(BaseEstimator, TransformerMixin):
    def __init__(self, low=25.0, high=44.0):
        self.low = low
        self.high = high
    
    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        return np.clip(np.asarray(X, dtype=float), self.low, self.high)
    
    def get_feature_names_out(self, input_features=None):
        """Propaga los nombres de features sin modificarlos."""
        return input_features

NUM_FEATURES = ['gestage_final']
CAT_OK = ['enrol_hiv_status_cat', 'momage_cat', 'educ_cat_n', 'marital_cat',
          'wealth_quintile', 'depression', 'mom_muac_cat', 'b1_sex', 'preterm']
CAT_NA = ['hfia_enr', 'parity', 'enrol_anemia', 'caesarean', 'sga', 'lbw']

num_pipe = Pipeline([('winsor', Winsorizer(25.0, 44.0)),
                     ('imputa', SimpleImputer(strategy='median')),
                     ('escala', StandardScaler())])
cat_ok_pipe = Pipeline([('imputa', SimpleImputer(strategy='most_frequent')),
                        ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))])
cat_na_pipe = Pipeline([('imputa', SimpleImputer(strategy='constant', fill_value='missing')),
                        ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))])

preprocessor = ColumnTransformer([('num', num_pipe, NUM_FEATURES),
                                  ('cat_ok', cat_ok_pipe, CAT_OK),
                                  ('cat_na', cat_na_pipe, CAT_NA)])
print('Preprocesador listo.')

Preprocesador listo.


## 6. Modelos baseline

In [16]:
models = {
    'logistic_regression': LogisticRegression(class_weight='balanced', max_iter=1000, random_state=SEED),
    'random_forest': RandomForestClassifier(n_estimators=300, max_depth=6, min_samples_leaf=5,
                                            class_weight='balanced', random_state=SEED, n_jobs=-1),
    'gradient_boosting': GradientBoostingClassifier(n_estimators=150, learning_rate=0.05,
                                                    max_depth=3, random_state=SEED),
}
print('Modelos baseline:', list(models.keys()))

Modelos baseline: ['logistic_regression', 'random_forest', 'gradient_boosting']


## 7. Entrenamiento y evaluacion

In [17]:
def evaluate_predictions(y_true, y_pred, y_proba):
    return {'accuracy': accuracy_score(y_true, y_pred),
            'precision': precision_score(y_true, y_pred, zero_division=0),
            'f1': f1_score(y_true, y_pred, zero_division=0),
            'sensitivity': recall_score(y_true, y_pred, zero_division=0),
            'pr_auc': average_precision_score(y_true, y_proba),
            'roc_auc': roc_auc_score(y_true, y_proba)}

def train_eval_model(model, X_tr, X_va, X_te, y_tr, y_va, y_te):
    pipe = Pipeline([('preprocessor', preprocessor), ('classifier', model)])
    pipe.fit(X_tr, y_tr)
    out = {}
    for tag, Xs, ys in [('train', X_tr, y_tr), ('val', X_va, y_va), ('test', X_te, y_te)]:
        out[tag] = evaluate_predictions(ys, pipe.predict(Xs), pipe.predict_proba(Xs)[:, 1])
    return pipe, out

print('Funciones de evaluacion listas.')

Funciones de evaluacion listas.


In [18]:
print('=' * 60)
print('HORIZONTE stunted_12')
print('=' * 60)
X_tr, X_va, X_te, y_tr, y_va, y_te, *_ = splits_12
results_12 = {}
for name, model in models.items():
    pipe, metrics = train_eval_model(model, X_tr, X_va, X_te, y_tr, y_va, y_te)
    results_12[name] = {'pipeline': pipe, 'metrics': metrics}
    print()
    print(name)
    print('  val :', {k: round(v, 3) for k, v in metrics['val'].items()})
    print('  test:', {k: round(v, 3) for k, v in metrics['test'].items()})

HORIZONTE stunted_12



logistic_regression
  val : {'accuracy': 0.612, 'precision': 0.176, 'f1': 0.24, 'sensitivity': 0.375, 'pr_auc': 0.37, 'roc_auc': 0.619}
  test: {'accuracy': 0.592, 'precision': 0.118, 'f1': 0.167, 'sensitivity': 0.286, 'pr_auc': 0.281, 'roc_auc': 0.531}

random_forest
  val : {'accuracy': 0.612, 'precision': 0.176, 'f1': 0.24, 'sensitivity': 0.375, 'pr_auc': 0.246, 'roc_auc': 0.567}
  test: {'accuracy': 0.592, 'precision': 0.118, 'f1': 0.167, 'sensitivity': 0.286, 'pr_auc': 0.29, 'roc_auc': 0.527}

gradient_boosting
  val : {'accuracy': 0.816, 'precision': 0.0, 'f1': 0.0, 'sensitivity': 0.0, 'pr_auc': 0.292, 'roc_auc': 0.634}
  test: {'accuracy': 0.796, 'precision': 0.286, 'f1': 0.286, 'sensitivity': 0.286, 'pr_auc': 0.321, 'roc_auc': 0.622}


In [19]:
print('=' * 60)
print('HORIZONTE stunted_24')
print('=' * 60)
X_tr, X_va, X_te, y_tr, y_va, y_te, *_ = splits_24
results_24 = {}
for name, model in models.items():
    pipe, metrics = train_eval_model(model, X_tr, X_va, X_te, y_tr, y_va, y_te)
    results_24[name] = {'pipeline': pipe, 'metrics': metrics}
    print()
    print(name)
    print('  val :', {k: round(v, 3) for k, v in metrics['val'].items()})
    print('  test:', {k: round(v, 3) for k, v in metrics['test'].items()})

HORIZONTE stunted_24

logistic_regression
  val : {'accuracy': 0.604, 'precision': 0.333, 'f1': 0.345, 'sensitivity': 0.357, 'pr_auc': 0.336, 'roc_auc': 0.59}
  test: {'accuracy': 0.551, 'precision': 0.348, 'f1': 0.421, 'sensitivity': 0.533, 'pr_auc': 0.445, 'roc_auc': 0.625}

random_forest
  val : {'accuracy': 0.625, 'precision': 0.375, 'f1': 0.4, 'sensitivity': 0.429, 'pr_auc': 0.305, 'roc_auc': 0.529}
  test: {'accuracy': 0.571, 'precision': 0.333, 'f1': 0.364, 'sensitivity': 0.4, 'pr_auc': 0.429, 'roc_auc': 0.618}

gradient_boosting
  val : {'accuracy': 0.604, 'precision': 0.273, 'f1': 0.24, 'sensitivity': 0.214, 'pr_auc': 0.296, 'roc_auc': 0.498}
  test: {'accuracy': 0.653, 'precision': 0.4, 'f1': 0.32, 'sensitivity': 0.267, 'pr_auc': 0.501, 'roc_auc': 0.667}


## 8. Comparacion de resultados

In [20]:
rows = []
for target, results in [('stunted_12', results_12), ('stunted_24', results_24)]:
    for model_name, info in results.items():
        for split, mets in info['metrics'].items():
            row = {'target': target, 'model': model_name, 'split': split}
            row.update(mets)
            rows.append(row)
results_df = pd.DataFrame(rows)
print('VALIDACION:')
display(results_df[results_df['split'] == 'val'].sort_values(['target', 'f1'], ascending=[True, False]))
print('TEST:')
display(results_df[results_df['split'] == 'test'].sort_values(['target', 'f1'], ascending=[True, False]))

VALIDACION:


,target,model,split,accuracy,precision,f1,sensitivity,pr_auc,roc_auc
1,stunted_12,logistic_regression,val,0.612245,0.176471,0.240000,0.375000,0.369913,0.618902
4,stunted_12,random_forest,val,0.612245,0.176471,0.240000,0.375000,0.246367,0.567073
7,stunted_12,gradient_boosting,val,0.816327,0.000000,0.000000,0.000000,0.291834,0.634146
13,stunted_24,random_forest,val,0.625000,0.375000,0.400000,0.428571,0.304561,0.529412
10,stunted_24,logistic_regression,val,0.604167,0.333333,0.344828,0.357143,0.335556,0.590336
16,stunted_24,gradient_boosting,val,0.604167,0.272727,0.240000,0.214286,0.296424,0.497899


TEST:


,target,model,split,accuracy,precision,f1,sensitivity,pr_auc,roc_auc
8,stunted_12,gradient_boosting,test,0.795918,0.285714,0.285714,0.285714,0.320799,0.622449
2,stunted_12,logistic_regression,test,0.591837,0.117647,0.166667,0.285714,0.281068,0.530612
5,stunted_12,random_forest,test,0.591837,0.117647,0.166667,0.285714,0.289928,0.527211
11,stunted_24,logistic_regression,test,0.551020,0.347826,0.421053,0.533333,0.445184,0.625490
14,stunted_24,random_forest,test,0.571429,0.333333,0.363636,0.400000,0.429013,0.617647
17,stunted_24,gradient_boosting,test,0.653061,0.400000,0.320000,0.266667,0.501064,0.666667


In [21]:
def best_by_val_f1(results):
    return max(results.items(), key=lambda kv: kv[1]['metrics']['val']['f1'])

best_model_12, best_info_12 = best_by_val_f1(results_12)
best_model_24, best_info_24 = best_by_val_f1(results_24)
best_pipeline_12 = best_info_12['pipeline']
best_pipeline_24 = best_info_24['pipeline']
print('Mejor 12m:', best_model_12, '| F1 val:', round(best_info_12['metrics']['val']['f1'], 4))
print('Mejor 24m:', best_model_24, '| F1 val:', round(best_info_24['metrics']['val']['f1'], 4))

Mejor 12m: logistic_regression | F1 val: 0.24
Mejor 24m: random_forest | F1 val: 0.4


## 9. Figuras para el reporte (se guardan en figures/ de la raiz)

In [22]:
X_te12, y_te12 = splits_12[2], splits_12[5]
X_te24, y_te24 = splits_24[2], splits_24[5]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
ConfusionMatrixDisplay.from_estimator(best_pipeline_12, X_te12, y_te12, ax=axes[0], cmap='Blues')
axes[0].set_title('Matriz de confusion test - stunted_12 (' + best_model_12 + ')')
ConfusionMatrixDisplay.from_estimator(best_pipeline_24, X_te24, y_te24, ax=axes[1], cmap='Blues')
axes[1].set_title('Matriz de confusion test - stunted_24 (' + best_model_24 + ')')
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig7_confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

proba_12 = best_pipeline_12.predict_proba(X_te12)[:, 1]
proba_24 = best_pipeline_24.predict_proba(X_te24)[:, 1]
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fpr1, tpr1, _ = roc_curve(y_te12, proba_12)
fpr2, tpr2, _ = roc_curve(y_te24, proba_24)
axes[0].plot(fpr1, tpr1, label='stunted_12')
axes[0].plot(fpr2, tpr2, label='stunted_24')
axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.4)
axes[0].set_title('Curvas ROC (test)')
axes[0].legend()
prec1, rec1, _ = precision_recall_curve(y_te12, proba_12)
prec2, rec2, _ = precision_recall_curve(y_te24, proba_24)
axes[1].plot(rec1, prec1, label='stunted_12')
axes[1].plot(rec2, prec2, label='stunted_24')
axes[1].set_title('Curvas Precision-Recall (test)')
axes[1].legend()
plt.tight_layout()
plt.savefig(FIG_DIR / 'fig7_curvas_roc_pr.png', dpi=150, bbox_inches='tight')
plt.show()

clf = best_pipeline_24.named_steps['classifier']
if hasattr(clf, 'feature_importances_'):
    names = best_pipeline_24.named_steps['preprocessor'].get_feature_names_out()
    imp = clf.feature_importances_
    def orig_var(n):
        pref, rest = n.split('__')
        return rest if pref == 'num' else rest.rsplit('_', 1)[0]
    serie = pd.Series(imp, index=[orig_var(n) for n in names]).groupby(level=0).sum().sort_values(ascending=False)
    top = serie.head(10)
    plt.figure(figsize=(9, 5))
    plt.barh(top.index[::-1], top.values[::-1], color='#e67e22')
    plt.title('Top 10 variables por importancia - mejor modelo 24m')
    plt.tight_layout()
    plt.savefig(FIG_DIR / 'fig7_importancia_variables.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('El mejor modelo 24m no expone feature_importances_; se omite esa figura.')

## 10. Guardado de modelos y metadatos (models/)

In [23]:
def file_md5(path):
    h = hashlib.md5()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(8192), b''):
            h.update(chunk)
    return h.hexdigest()

data_hash = file_md5(DATA_PATH)
print('MD5 del dataset:', data_hash)

model_12_path = MODELS_DIR / ('model_stunting_12m_' + best_model_12 + '.joblib')
model_24_path = MODELS_DIR / ('model_stunting_24m_' + best_model_24 + '.joblib')
joblib.dump(best_pipeline_12, model_12_path)
joblib.dump(best_pipeline_24, model_24_path)

metadata = {'data_source': str(DATA_PATH), 'data_hash': data_hash, 'features': FEATURES,
            'targets': TARGETS, 'seed': SEED, 'best_model_12': best_model_12,
            'best_model_24': best_model_24, 'metrics_12': best_info_12['metrics'],
            'metrics_24': best_info_24['metrics']}
with open(MODELS_DIR / 'model_metadata.json', 'w', encoding='utf-8') as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

print('Guardado:', model_12_path.name)
print('Guardado:', model_24_path.name)
print('Guardado: model_metadata.json')

MD5 del dataset: 493798e633bb5f8384b6ad87108df862
Guardado: model_stunting_12m_logistic_regression.joblib
Guardado: model_stunting_24m_random_forest.joblib
Guardado: model_metadata.json


## 11. Logging opcional en MLflow

In [24]:
if MLFLOW_AVAILABLE:
    mlflow.set_experiment('stunting_baseline_model_dataset')
    runs = [('stunted_12', best_model_12, best_info_12, best_pipeline_12),
            ('stunted_24', best_model_24, best_info_24, best_pipeline_24)]
    for target, best_name, best_info, pipe in runs:
        with mlflow.start_run(run_name='baseline_' + target + '_' + best_name):
            mlflow.log_param('target', target)
            mlflow.log_param('data_hash', data_hash)
            mlflow.log_param('features_type', '16_basales')
            mlflow.log_param('best_model', best_name)
            mlflow.log_param('seed', SEED)
            for split, mets in best_info['metrics'].items():
                for k, v in mets.items():
                    mlflow.log_metric(split + '_' + k, v)
            try:
                mlflow.sklearn.log_model(pipe, name='model')
            except TypeError:
                mlflow.sklearn.log_model(pipe, artifact_path='model')
    print('MLflow logging completado.')
else:
    print('[INFO] Sin MLflow; logging omitido.')

[INFO] Sin MLflow; logging omitido.


## 12. Resumen y proximos pasos

- Baseline tiempo-cero (16 features basales) entrenado, evaluado y versionado para ambos horizontes.
- Proximo experimento: agregar features postnales tempranas (Z-scores week-3/month-3 y pendiente dLAZ) y comparar en MLflow.
- Despues: tuning de hiperparametros y definicion del punto de operacion (umbral) para el tablero.